# DPO from Scratch — Colab walkthrough

Fine-tune `Qwen2.5-0.5B-Instruct` on `Anthropic/hh-rlhf` with DPO. ~30-40 min on a T4 with LoRA.

**Runtime → Change runtime type → T4 GPU** before running. If you don't have access to a GPU, you can still run the cells but training will be unusably slow.

Companion blog post: `BLOG.md` in the repo.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade transformers>=4.45.0 trl>=0.11.0 peft>=0.13.0 datasets>=3.0.0 accelerate>=0.34.0 bitsandbytes
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Prepare data

HH-RLHF stores each example as two complete dialogues that diverge only at the final assistant turn. We split that final turn off so `prompt` is the shared context (formatted into Qwen's chat template) and `chosen`/`rejected` are just the candidate responses.

In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

HH_TURN_RE = re.compile(r'\n\n(Human|Assistant): ')

def split_prompt_response(text):
    idx = text.rfind('\n\nAssistant: ')
    prompt = text[:idx + len('\n\nAssistant: ')]
    response = text[idx + len('\n\nAssistant: '):]
    return prompt, response

def hh_to_messages(prompt_text):
    body = re.sub(r'\n\nAssistant:\s*$', '', prompt_text.strip())
    parts = HH_TURN_RE.split('\n\n' + body)
    return [
        {'role': 'user' if role == 'Human' else 'assistant', 'content': content.strip()}
        for role, content in zip(parts[1::2], parts[2::2])
    ]

def format_for_dpo(example):
    prompt_text, chosen_resp = split_prompt_response(example['chosen'])
    _, rejected_resp = split_prompt_response(example['rejected'])
    messages = hh_to_messages(prompt_text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {'prompt': prompt, 'chosen': chosen_resp.strip(), 'rejected': rejected_resp.strip()}

raw = load_dataset('Anthropic/hh-rlhf')
train = raw['train'].shuffle(seed=0).select(range(8000)).map(format_for_dpo, remove_columns=raw['train'].column_names)
test = raw['test'].shuffle(seed=0).select(range(500)).map(format_for_dpo, remove_columns=raw['test'].column_names)

def keep(ex):
    return (len(ex['prompt']) < 4000 and 5 < len(ex['chosen']) < 2000 and 5 < len(ex['rejected']) < 2000)
train, test = train.filter(keep), test.filter(keep)

print(f'Train: {len(train)} | Test: {len(test)}')
print('\n--- Example ---')
print('PROMPT:', train[0]['prompt'][:400], '...')
print('\nCHOSEN  :', train[0]['chosen'][:200])
print('\nREJECTED:', train[0]['rejected'][:200])

## 3. Train (DPO + LoRA)

LoRA lets us skip a second model in memory — TRL recovers `π_ref` by disabling the adapter on the fly. β=0.1 is the canonical choice; see the blog post for what happens when you push it.

Expected runtime: ~30 min on a T4.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import DPOConfig, DPOTrainer

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, attn_implementation='eager')
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

cfg = DPOConfig(
    output_dir='./qwen05b-dpo',
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    max_length=1024, max_prompt_length=512,
    logging_steps=10,
    eval_strategy='steps', eval_steps=200,
    save_strategy='steps', save_steps=400, save_total_limit=2,
    warmup_ratio=0.1, lr_scheduler_type='cosine',
    fp16=True,
    remove_unused_columns=False, report_to='none', seed=42,
)

trainer = DPOTrainer(
    model=model, ref_model=None,
    args=cfg,
    train_dataset=train,
    eval_dataset=test.select(range(256)),
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model('./qwen05b-dpo')
tokenizer.save_pretrained('./qwen05b-dpo')

## 4. Evaluate — does the implicit reward margin go up?

On held-out HH pairs, compute `(log π_θ(y_w|x) − log π_ref(y_w|x)) − (log π_θ(y_l|x) − log π_ref(y_l|x))` and check that the margin is positive on average. This is the metric DPO is *directly* optimising; if it doesn't improve, training failed.

In [ ]:
from peft import PeftModel

device = 'cuda'
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device).eval()
policy_backbone = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device)
policy = PeftModel.from_pretrained(policy_backbone, './qwen05b-dpo').to(device).eval()

@torch.no_grad()
def cond_logprob(model, prompt, response):
    prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    full_ids = tokenizer(prompt + response, return_tensors='pt').input_ids.to(device)
    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100
    out = model(full_ids, labels=labels)
    n = (labels != -100).sum().item()
    return -float(out.loss.item()) * n if n else 0.0

margins, accs = [], []
for i, ex in enumerate(test.select(range(200))):
    lp_c_p = cond_logprob(policy, ex['prompt'], ex['chosen'])
    lp_r_p = cond_logprob(policy, ex['prompt'], ex['rejected'])
    lp_c_b = cond_logprob(base,   ex['prompt'], ex['chosen'])
    lp_r_b = cond_logprob(base,   ex['prompt'], ex['rejected'])
    margin = (lp_c_p - lp_c_b) - (lp_r_p - lp_r_b)
    margins.append(margin); accs.append(1.0 if margin > 0 else 0.0)

print(f'n={len(margins)}  mean_margin={sum(margins)/len(margins):+.3f}  pref_accuracy={sum(accs)/len(accs):.2%}')

## 5. Side-by-side generations

The numerical metric tells you the loss moved in the right direction. Sampling tells you what that *looks* like to a human reader.

In [ ]:
PROMPTS = [
    'My friend just told me they\'re feeling really down. What should I say?',
    'Explain why airplanes can fly to a curious 10-year-old.',
    'Should I tell my manager that I\'m interviewing elsewhere? Pros and cons.',
    'Write a haiku about debugging a memory leak at 2am.',
]

@torch.no_grad()
def gen(model, prompt, seed=0):
    torch.manual_seed(seed)
    text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt').to(device)
    out = model.generate(**inp, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

for i, p in enumerate(PROMPTS, 1):
    print(f'\n=== Prompt {i}: {p} ===')
    print(f'\n[BASE]\n{gen(base, p, seed=i)}')
    print(f'\n[DPO]\n{gen(policy, p, seed=i)}')
    print('-' * 80)

## Done

Read `BLOG.md` for what to look for in the metrics (especially: both chosen and rejected log-probs typically go *down*, and that's normal).

Things to try next:
- β=0.01 vs β=1.0 — what happens to KL and to the loss?
- SFT-on-chosen baseline — how much of DPO's win is just learning to imitate the chosen responses?
- Length-controlled preference accuracy — is DPO learning quality or just length?